# Part II: Linear systems: solutions to the code exercises

Worked solutions to the code exercises from [*Matrices in Practice*](https://trgoodale.github.io/matrix-book/appendices/solutions-2.html), by Tom Goodale.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/trgoodale/matrix-book/blob/main/code/python/solutions-part-2.ipynb)

Run the cells in order, from the top: later cells use variables set in earlier ones. Try each exercise yourself first. Change the numbers and run them again to see what happens. The code needs NumPy, SciPy and Matplotlib (and pandas in a few chapters), all of which Colab already has.

Code licensed under the [MIT licence](https://trgoodale.github.io/matrix-book/LICENSE.md).

## Chapter 10: Systems of linear equations

### Exercise 7

In Example 1, find the mixture for a new alloy of 100 kg containing 65 kg of copper, 20 kg of zinc and 15 kg of tin. Then try 55 kg of copper, 30 kg of zinc and 15 kg of tin. What goes wrong, and what does it mean for the workshop?

In [1]:
import numpy as np

A = np.array([[0.6, 0.8, 0.5],
              [0.3, 0.1, 0.2],
              [0.1, 0.1, 0.3]])
print(np.linalg.solve(A, [65, 20, 15]))
print(np.linalg.solve(A, [55, 30, 15]))

[37.5 37.5 25. ]
[ 87.5 -12.5  25. ]


The first alloy needs 37.5 kg of alloy 1, 37.5 kg of alloy 2 and 25 kg of alloy 3. The second "needs" $-12.5$ kg of alloy 2, which is impossible: the workshop can't add a negative amount. The equations have a solution, but it has no physical meaning, so this alloy can't be made from these three scraps. In the column picture, $\mathbf{b}$ can be made from the columns, but not with amounts that are all positive.

## Chapter 11: Gaussian elimination

### Exercise 6

Add a counter to the `gaussian_elimination` function that counts every multiplication, division, addition and subtraction. Run it on random systems with $n = 10, 20, 40, 80$, and compare the counts with $\tfrac{2}{3}n^3$.

One way to count operations:

In [2]:
import numpy as np

def elimination_count(A, b):
    """Gaussian elimination that also counts arithmetic operations."""
    A = np.array(A, dtype=float)
    b = np.array(b, dtype=float)
    n = len(b)
    ops = 0
    for k in range(n - 1):
        for i in range(k + 1, n):
            m = A[i, k] / A[k, k]
            A[i, k:] -= m * A[k, k:]
            b[i] -= m * b[k]
            ops += 1 + 2 * (n - k) + 2      # divide, row update, right-hand side
    x = np.zeros(n)
    for i in range(n - 1, -1, -1):
        x[i] = (b[i] - A[i, i + 1:] @ x[i + 1:]) / A[i, i]
        ops += 2 * (n - i - 1) + 2          # dot product, subtract, divide
    return x, ops

rng = np.random.default_rng(0)
for n in [10, 20, 40, 80]:
    _, ops = elimination_count(rng.standard_normal((n, n)), rng.standard_normal(n))
    print(f"n = {n:3d}   operations = {ops:7d}   2n^3/3 = {2 * n**3 / 3:9.0f}   ratio = {ops / (2 * n**3 / 3):.2f}")

n =  10   operations =     905   2n^3/3 =       667   ratio = 1.36
n =  20   operations =    6310   2n^3/3 =      5333   ratio = 1.18
n =  40   operations =   46620   2n^3/3 =     42667   ratio = 1.09
n =  80   operations =  357240   2n^3/3 =    341333   ratio = 1.05


The ratio approaches 1 as $n$ grows. For small systems the lower-order terms, such as the $n^2$ operations of back substitution, still matter.

## Chapter 12: Gauss–Jordan elimination and rank

### Exercise 8

In photosynthesis, plants turn carbon dioxide and water into glucose and oxygen: $a\,\mathrm{CO_2} + b\,\mathrm{H_2O} \to c\,\mathrm{C_6H_{12}O_6} + d\,\mathrm{O_2}$. Write the atom balances for carbon, hydrogen and oxygen as a homogeneous system (as in Example 3), find its reduced row echelon form with code, and use it to balance the equation with the smallest whole numbers.

The balances are: carbon, $a = 6c$; hydrogen, $2b = 12c$; oxygen, $2a + b = 6c + 2d$. As a homogeneous system in $a, b, c, d$:

In [3]:
import numpy as np

def rref(M, tol=1e-10):
    R = np.array(M, dtype=float)
    m, n = R.shape
    pivots, r = [], 0
    for c in range(n):
        if r == m:
            break
        p = r + np.argmax(abs(R[r:, c]))
        if abs(R[p, c]) < tol:
            continue
        R[[r, p]] = R[[p, r]]
        R[r] /= R[r, c]
        for i in range(m):
            if i != r:
                R[i] -= R[i, c] * R[r]
        pivots.append(c)
        r += 1
    return R, pivots

M = [[1, 0, -6, 0],      # carbon
     [0, 2, -12, 0],     # hydrogen
     [2, 1, -6, -2]]     # oxygen
R, piv = rref(M)
print(np.round(R, 4) + 0)

[[ 1.      0.      0.     -1.    ]
 [ 0.      1.      0.     -1.    ]
 [ 0.      0.      1.     -0.1667]]


The free variable is $d$, and the rows say $a = d$, $b = d$ and $c = d/6$. The smallest whole numbers come from $d = 6$:
$$
6\,\mathrm{CO_2} + 6\,\mathrm{H_2O} \longrightarrow \mathrm{C_6H_{12}O_6} + 6\,\mathrm{O_2}.
$$

## Chapter 13: LU decomposition

### Exercise 7

Write a function `forward_substitution(L, b)` that solves $\mathbf{L}\mathbf{y} = \mathbf{b}$ for a lower triangular $\mathbf{L}$ with 1s on its diagonal. Use it, with the back substitution from [Chapter 11](https://trgoodale.github.io/matrix-book/part-2/11-gaussian-elimination.html), to solve the systems in Exercise 2, and check your answers against a library solver.

In [4]:
import numpy as np

def forward_substitution(L, b):
    """Solve Ly = b, for L lower triangular with 1s on its diagonal."""
    n = len(b)
    y = np.zeros(n)
    for i in range(n):
        y[i] = b[i] - L[i, :i] @ y[:i]
    return y

def back_substitution(U, y):
    """Solve Ux = y, for U upper triangular."""
    n = len(y)
    x = np.zeros(n)
    for i in range(n - 1, -1, -1):
        x[i] = (y[i] - U[i, i + 1:] @ x[i + 1:]) / U[i, i]
    return x

L = np.array([[1, 0, 0], [2, 1, 0], [3, 2, 1]])
U = np.array([[1, 1, 1], [0, 1, 2], [0, 0, 1]])
A = L @ U
for b in ([3, 9, 16], [0, 1, 3]):
    x = back_substitution(U, forward_substitution(L, b))
    print(x, np.allclose(x, np.linalg.solve(A, b)))

[1. 1. 1.] True
[ 0. -1.  1.] True


## Part II: Linear systems review

### Exercise 8

For the matrix in exercise 4, factorise once (with `lu_factor` in Python or `lu` in MATLAB), then solve for the right-hand sides $(4, 10, 24)^{\mathsf{T}}$, $\mathbf{e}_{1}$, $\mathbf{e}_{2}$ and $\mathbf{e}_{3}$. Use the last three solutions to build $\mathbf{A}^{-1}$, and compare it with the built-in inverse.

*Uses [Chapter 13](https://trgoodale.github.io/matrix-book/part-2/13-lu-decomposition.html) and [Chapter 7](https://trgoodale.github.io/matrix-book/part-1/07-inverses.html).*

In [5]:
import numpy as np
import scipy.linalg as sl

A = np.array([[2.0, 1, 1], [4, 3, 3], [8, 7, 9]])
f = sl.lu_factor(A)                       # factorise once
print(sl.lu_solve(f, [4, 10, 24]))

# Solving for e1, e2 and e3 gives the columns of the inverse
A_inv = np.column_stack([sl.lu_solve(f, e) for e in np.eye(3)])
print(np.round(A_inv, 4))
print("Agrees with inv(A):", np.allclose(A_inv, np.linalg.inv(A)))

[1. 1. 1.]
[[ 1.5 -0.5  0. ]
 [-3.   2.5 -0.5]
 [ 1.  -1.5  0.5]]
Agrees with inv(A): True


This is how software finds an inverse when one is really needed: one factorisation, then one cheap solve per column.